
## Naive Bayes from Scratch: Fruit Quality Grading

## Dataset

Each row is one inspected fruit sample. All predictors use text-based discrete values.

| Property | Possible values |
|---|---|
| `skin_color` | `green`, `yellow`, `golden` |
| `firmness` | `soft`, `medium`, `firm` |
| `size_group` | `small`, `medium`, `large` |
| `blemish_level` | `none`, `few`, `many` |
| `sugar_level` | `low`, `medium`, `high` |

The target is `quality_grade`. The `set_name` column fixes the training and test sets.

In [1]:
%pip install numpy pandas


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

inspection_columns = [
    "skin_color",
    "firmness",
    "size_group",
    "blemish_level",
    "sugar_level",
    "quality_grade",
    "set_name",
]

inspection_rows = [
    ["golden", "firm", "large", "none", "high", "grade_a", "train"],
    ["yellow", "firm", "medium", "none", "high", "grade_a", "train"],
    ["golden", "medium", "large", "few", "high", "grade_a", "train"],
    ["yellow", "firm", "large", "few", "medium", "grade_a", "train"],
    ["golden", "firm", "medium", "few", "high", "grade_a", "train"],
    ["yellow", "medium", "large", "none", "medium", "grade_a", "train"],
    ["golden", "firm", "small", "none", "high", "grade_a", "train"],
    ["yellow", "firm", "medium", "few", "medium", "grade_a", "train"],
    ["golden", "medium", "medium", "none", "high", "grade_a", "train"],
    ["yellow", "firm", "large", "none", "high", "grade_a", "train"],
    ["golden", "firm", "large", "few", "medium", "grade_a", "train"],
    ["yellow", "medium", "medium", "none", "high", "grade_a", "train"],
    ["green", "soft", "small", "many", "low", "grade_b", "train"],
    ["green", "medium", "small", "few", "low", "grade_b", "train"],
    ["yellow", "soft", "medium", "many", "medium", "grade_b", "train"],
    ["green", "soft", "medium", "few", "low", "grade_b", "train"],
    ["yellow", "medium", "small", "many", "low", "grade_b", "train"],
    ["green", "firm", "small", "few", "medium", "grade_b", "train"],
    ["yellow", "soft", "small", "few", "low", "grade_b", "train"],
    ["green", "medium", "medium", "many", "low", "grade_b", "train"],
    ["yellow", "soft", "large", "many", "medium", "grade_b", "train"],
    ["green", "soft", "small", "none", "low", "grade_b", "train"],
    ["yellow", "medium", "medium", "many", "medium", "grade_b", "train"],
    ["green", "medium", "large", "few", "low", "grade_b", "train"],
    ["golden", "firm", "large", "none", "high", "grade_a", "test"],
    ["yellow", "firm", "medium", "few", "high", "grade_a", "test"],
    ["golden", "medium", "small", "none", "high", "grade_a", "test"],
    ["yellow", "medium", "large", "none", "medium", "grade_a", "test"],
    ["green", "soft", "small", "many", "low", "grade_b", "test"],
    ["yellow", "soft", "medium", "many", "low", "grade_b", "test"],
    ["green", "medium", "large", "few", "low", "grade_b", "test"],
    ["yellow", "medium", "small", "many", "medium", "grade_b", "test"],
]

fruit_inspections = pd.DataFrame(inspection_rows, columns=inspection_columns)
fruit_inspections

,skin_color,firmness,size_group,blemish_level,sugar_level,quality_grade,set_name
0,golden,firm,large,none,high,grade_a,train
1,yellow,firm,medium,none,high,grade_a,train
2,golden,medium,large,few,high,grade_a,train
3,yellow,firm,large,few,medium,grade_a,train
4,golden,firm,medium,few,high,grade_a,train
5,yellow,medium,large,none,medium,grade_a,train
6,golden,firm,small,none,high,grade_a,train
7,yellow,firm,medium,few,medium,grade_a,train
8,golden,medium,medium,none,high,grade_a,train
9,yellow,firm,large,none,high,grade_a,train


In [3]:
fruit_properties = [
    "skin_color",
    "firmness",
    "size_group",
    "blemish_level",
    "sugar_level",
]

training_inspections = fruit_inspections[
    fruit_inspections["set_name"] == "train"
].reset_index(drop=True)
testing_inspections = fruit_inspections[
    fruit_inspections["set_name"] == "test"
].reset_index(drop=True)

fruit_train = training_inspections[fruit_properties]
grade_train = training_inspections["quality_grade"]
fruit_test = testing_inspections[fruit_properties]
grade_test = testing_inspections["quality_grade"]
grade_names = np.sort(grade_train.unique())

print("Training inspections:", fruit_train.shape)
print("Testing inspections: ", fruit_test.shape)
print(grade_train.value_counts())

Training inspections: (24, 5)
Testing inspections:  (8, 5)
quality_grade
grade_a    12
grade_b    12
Name: count, dtype: int64


## Label summary

Complete `make_label_summary`. Return a DataFrame indexed by the class labels with exactly two columns: `frequency` and `prior`. Calculate each prior as the class frequency divided by the number of training samples.

In [4]:
def make_label_summary(labels):
    """Summarize class frequencies and prior probabilities.

    Parameters
    ----------
    labels : pandas.Series
        Training class labels.

    Returns
    -------
    pandas.DataFrame
        Class-indexed table with frequency and prior columns.
    """
    summary_rows = []

    for grade in np.sort(labels.unique()):
        grade_frequency = (labels == grade).sum()
        grade_prior = grade_frequency / len(labels)
        summary_rows.append(
            {
                "quality_grade": grade,
                "frequency": grade_frequency,
                "prior": grade_prior,
            }
        )

    return pd.DataFrame(summary_rows).set_index("quality_grade")


label_summary = make_label_summary(grade_train)
label_summary

,frequency,prior
quality_grade,,
grade_a,12,0.5
grade_b,12,0.5


## Nested probability model

Complete `learn_probability_tree`. The return value must support a lookup in this order:

```python
probability_tree[class_label][property_name][observed_value]
```

Use the general discrete-value formula with additive smoothing:

$$P(X_j=v\mid C=c)=\frac{N_{j,v,c}+\alpha}{N_c+\alpha K_j}$$

The number of categories $K_j$ must be determined separately for each property from the complete training set.

In [5]:
def learn_probability_tree(samples, labels, alpha=1.0):
    """Learn smoothed likelihoods in a three-level dictionary.

    Parameters
    ----------
    samples : pandas.DataFrame
        Discrete training properties.
    labels : pandas.Series
        Training labels aligned with samples.
    alpha : float
        Laplace smoothing parameter.

    Returns
    -------
    dict
        Nested mapping: class, property, observed value, probability.
    """
    property_values = {
        property_name: np.sort(samples[property_name].unique())
        for property_name in samples.columns
    }
    probability_tree = {}

    for grade in np.sort(labels.unique()):
        grade_samples = samples[labels == grade]
        probability_tree[grade] = {}

        for property_name, possible_values in property_values.items():
            probability_tree[grade][property_name] = {}

            for possible_value in possible_values:
                occurrence_count = (grade_samples[property_name] == possible_value).sum()
                smoothed_probability = (occurrence_count + alpha) / (len(grade_samples) 
                                                                     + alpha * len(possible_values))
                probability_tree[grade][property_name][possible_value] = (
                    smoothed_probability
                )

    return probability_tree


probability_tree = learn_probability_tree(
    fruit_train, grade_train, alpha=1.0
)
probability_tree

{'grade_a': {'skin_color': {'golden': np.float64(0.4666666666666667),
   'green': np.float64(0.06666666666666667),
   'yellow': np.float64(0.4666666666666667)},
  'firmness': {'firm': np.float64(0.6),
   'medium': np.float64(0.3333333333333333),
   'soft': np.float64(0.06666666666666667)},
  'size_group': {'large': np.float64(0.4666666666666667),
   'medium': np.float64(0.4),
   'small': np.float64(0.13333333333333333)},
  'blemish_level': {'few': np.float64(0.4),
   'many': np.float64(0.06666666666666667),
   'none': np.float64(0.5333333333333333)},
  'sugar_level': {'high': np.float64(0.6),
   'low': np.float64(0.06666666666666667),
   'medium': np.float64(0.3333333333333333)}},
 'grade_b': {'skin_color': {'golden': np.float64(0.06666666666666667),
   'green': np.float64(0.5333333333333333),
   'yellow': np.float64(0.4)},
  'firmness': {'firm': np.float64(0.13333333333333333),
   'medium': np.float64(0.4),
   'soft': np.float64(0.4666666666666667)},
  'size_group': {'large': np.float

## Batch grading

Complete `classify_samples`. For every sample, calculate one log-score per grade from `label_summary` and `probability_tree`. The function must return two objects:

1. A DataFrame of log-scores named `fruit_logbook`.
2. A NumPy array of selected grades named `assigned_grades`.

Select the grade with the largest score.

In [6]:
def classify_samples(samples, class_names, summary, probability_model):
    """Score and classify a batch of fruit samples.

    Parameters
    ----------
    samples : pandas.DataFrame
        Fruit samples to classify.
    class_names : array-like
        Candidate quality grades.
    summary : pandas.DataFrame
        Table containing the prior for each grade.
    probability_model : dict
        Nested conditional-probability dictionary.

    Returns
    -------
    tuple
        A log-score DataFrame and a one-dimensional prediction array.
    """
    score_rows = []
    selected_grades = []

    for sample_index, sample in samples.iterrows():
        sample_scores = {}

        for grade in class_names:
            grade_score = np.log(summary.loc[grade, "prior"])

            for property_name, observed_value in sample.items():
                likelihood = probability_model[grade][property_name].get(observed_value, 0)
                grade_score = grade_score + np.log(likelihood) if likelihood > 0 else float('-inf')

            sample_scores[grade] = grade_score

        score_rows.append(sample_scores)
        selected_grades.append(max(sample_scores, key=sample_scores.get))

    return (
        pd.DataFrame(score_rows, index=samples.index),
        np.asarray(selected_grades),
    )


fruit_logbook, assigned_grades = classify_samples(
    fruit_test, grade_names, label_summary, probability_tree
)

grading_report = fruit_test.copy()
grading_report["actual_grade"] = grade_test.to_numpy()
grading_report["assigned_grade"] = assigned_grades
display(fruit_logbook)
display(grading_report)

,grade_a,grade_b
0,-3.867687,-11.748492
1,-4.309520,-8.347294
2,-5.708237,-9.802581
3,-5.043261,-7.248682
4,-13.540251,-4.119002
5,-10.495729,-4.743156
6,-8.886291,-5.274601
7,-8.375465,-5.148621


,skin_color,firmness,size_group,blemish_level,sugar_level,actual_grade,assigned_grade
0,golden,firm,large,none,high,grade_a,grade_a
1,yellow,firm,medium,few,high,grade_a,grade_a
2,golden,medium,small,none,high,grade_a,grade_a
3,yellow,medium,large,none,medium,grade_a,grade_a
4,green,soft,small,many,low,grade_b,grade_b
5,yellow,soft,medium,many,low,grade_b,grade_b
6,green,medium,large,few,low,grade_b,grade_b
7,yellow,medium,small,many,medium,grade_b,grade_b


## Grade outcome tally

Complete `tally_grade_pairs`. Return a nested dictionary in the form `tally[actual_grade][assigned_grade] = count`. Every possible pair must appear, even when its count is zero. Then calculate `grade_accuracy` from scratch.

In [7]:
def tally_grade_pairs(actual_grades, predicted_grades, grade_order):
    """Count every actual-grade and predicted-grade combination.

    Parameters
    ----------
    actual_grades : array-like
        Actual quality grades.
    predicted_grades : array-like
        Model-assigned quality grades.
    grade_order : array-like
        Complete ordered collection of grade labels.

    Returns
    -------
    dict
        Nested counts keyed first by actual grade.
    """
    tally = {
        actual: {predicted: 0 for predicted in grade_order}
        for actual in grade_order
    }

    for actual, predicted in zip(actual_grades, predicted_grades):
        tally[actual][predicted] += 1

    return tally


grade_tally = tally_grade_pairs(
    grade_test, assigned_grades, grade_names
)

grade_accuracy = np.mean(grade_test.to_numpy() == assigned_grades)

print(f"Grade accuracy: {grade_accuracy:.3f}")
display(pd.DataFrame(grade_tally).T)

Grade accuracy: 1.000


,grade_a,grade_b
grade_a,4,0
grade_b,0,4
